In [1]:
from pathlib import Path
import re

import pandas as pd
from pandas import DataFrame

from config import PATHS
from helpers import rename_patient

In [2]:
# Paths
# on server:
# in_path = PATHS.per_model_comparison_table.pickle  # on server
# out_path = Path("~/thesis_data/tables/model_eval.tex").expanduser()

# local:
in_path = Path('~/thesis_files/statistical_results/.per_model.pkl').expanduser()  # local
out_path = Path('~/Developer/MastersThesis/src/tables/model_eval_metrics.tex').expanduser()

In [21]:
per_model = pd.read_pickle(in_path)
fake_mask = per_model.index.get_level_values('patient').str.contains('FAKE')
per_model = per_model[~fake_mask]
per_model

metric                 best_threshold            rel_tifw            \
split                           train      test     train      test   
patient       model                                                   
competition-1 CNN            0.730252  0.348021  0.150953  0.407381   
              ensemble       0.548969  0.420712  0.368110  0.494720   
competition-2 CNN            0.592629  0.503402  0.196632  0.330344   
              ensemble       0.604010  0.561291  0.323201  0.397423   
competition-3 CNN            0.779933  0.696605  0.142371  0.263264   
              ensemble       0.654698  0.622529  0.217113  0.285408   
U002-DE01-01  CNN            0.582112  0.634063  0.226472  0.165048   
              ensemble       0.567867  0.544376  0.266903  0.239180   
U002-DE01-03  CNN            0.780604  0.674600  0.107185  0.373027   
              ensemble       0.574479  0.525727  0.186428  0.426924   
U002-DE01-04  CNN            0.599625  0.559766  0.269078  0.472266   
              ensemble       0.526974  0.505274  0.430727  0.600760   
U002-DE01-05  CNN            0.756170  0.598670  0.065411  0.217563   
              ensemble       0.587826  0.586498  0.171846  0.189639   
U002-DE01-07  CNN            0.736176  0.701969  0.034486  0.183846   
              ensemble       0.545029  0.536581  0.282319  0.457661   
U002-DE01-12  CNN            0.666480  0.465285  0.072165  0.394819   
              ensemble       0.563852  0.561054  0.420331  0.482148   
U002-DE01-15  CNN            0.967000  0.868920  0.047427  0.345278   
              ensemble       0.561869  0.539433  0.315991  0.413938   
U002-DE01-16  CNN            0.812213  0.752847  0.130959  0.350571   
              ensemble       0.563557  0.559030  0.168770  0.215980   
U002-DE01-17  CNN            0.667585  0.528467  0.156727  0.358979   
              ensemble       0.609517  0.604358  0.345519  0.305847   

metric                 rel_szrs_pred           event_based_f1            \
split                          train      test          train      test   
patient       model                                                       
competition-1 CNN           0.794118  0.541667       0.820664  0.565998   
              ensemble      0.852941  0.500000       0.725961  0.502626   
competition-2 CNN           0.967742  0.863636       0.877928  0.754376   
              ensemble      0.838710  0.636364       0.749106  0.619009   
competition-3 CNN           0.945946  0.846154       0.899625  0.787663   
              ensemble      0.837838  0.961538       0.809431  0.819874   
U002-DE01-01  CNN           0.880342  0.794872       0.823486  0.814419   
              ensemble      0.803419  0.782051       0.766649  0.771290   
U002-DE01-03  CNN           0.852941  0.666667       0.872423  0.646211   
              ensemble      0.735294  0.444444       0.772455  0.500630   
U002-DE01-04  CNN           0.783784  0.560000       0.756430  0.543388   
              ensemble      0.729730  0.400000       0.639591  0.399620   
U002-DE01-05  CNN           0.800000  0.857143       0.862073  0.818088   
              ensemble      0.775000  0.785714       0.800696  0.797847   
U002-DE01-07  CNN           1.000000  0.666667       0.982454  0.733875   
              ensemble      0.857143  0.833333       0.781237  0.657059   
U002-DE01-12  CNN           0.777778  0.666667       0.846206  0.634438   
              ensemble      0.833333  0.750000       0.683732  0.612672   
U002-DE01-15  CNN           0.769231  0.700000       0.851141  0.676604   
              ensemble      0.846154  0.600000       0.756490  0.592949   
U002-DE01-16  CNN           1.000000  0.625000       0.929932  0.636981   
              ensemble      0.750000  0.750000       0.788529  0.766633   
U002-DE01-17  CNN           0.833333  0.777778       0.838274  0.702808   
              ensemble      0.833333  0.777778       0.733157  0.733590   

metric                 precision              recall    

In [22]:
# n: numeric table
n: DataFrame = per_model.copy()

# Change rel TIFW to EB Specificity
n['rel_tifw'] = 1 - n['rel_tifw']
n = n.rename(columns={'rel_tifw': 'eb_specificity'})

# Put all metrics in percent
ALPHA = 0.05 * 100
n = n * 100

# Abbreviate patients
idx_df = n.index.to_frame(index=False)
idx_df['patient'] = idx_df['patient'].map(rename_patient)
n.index = pd.MultiIndex.from_frame(idx_df)

p_hm = n['p_hanley_mcneil']

n = n.drop(columns=[
    # 'best_threshold',
    'p_hanley_mcneil',
    'precision',
    'recall',
], level='metric')

n

metric           best_threshold            eb_specificity             \
split                     train       test          train       test   
patient model                                                          
C01     CNN           73.025203  34.802124      84.904749  59.261907   
        ensemble      54.896939  42.071170      63.188951  50.527995   
C02     CNN           59.262931  50.340164      80.336831  66.965594   
        ensemble      60.400975  56.129110      67.679937  60.257675   
C03     CNN           77.993321  69.660503      85.762928  73.673615   
        ensemble      65.469843  62.252885      78.288672  71.459247   
U01     CNN           58.211178  63.406336      77.352808  83.495247   
        ensemble      56.786734  54.437566      73.309743  76.082036   
U03     CNN           78.060365  67.460024      89.281508  62.697324   
        ensemble      57.447898  52.572662      81.357152  57.307598   
U04     CNN           59.962547  55.976635      73.092182  52.773398   
        ensemble      52.697384  50.527406      56.927266  39.924036   
U05     CNN           75.617027  59.867007      93.458931  78.243705   
        ensemble      58.782554  58.649826      82.815389  81.036060   
U07     CNN           73.617613  70.196933      96.551388  81.615423   
        ensemble      54.502922  53.658110      71.768144  54.233871   
U12     CNN           66.648036  46.528545      92.783505  60.518097   
        ensemble      56.385243  56.105387      57.966950  51.785167   
U15     CNN           96.700019  86.892021      95.257321  65.472151   
        ensemble      56.186914  53.943300      68.400879  58.606229   
U16     CNN           81.221294  75.284684      86.904095  64.942928   
        ensemble      56.355727  55.902994      83.123035  78.401985   
U17     CNN           66.758549  52.846658      84.327275  64.102088   
        ensemble      60.951704  60.435760      65.448138  69.415313   

metric           rel_szrs_pred            event_based_f1             \
split                    train       test          train       test   
patient model                                                         
C01     CNN          79.411765  54.166667      82.066443  56.599847   
        ensemble     85.294118  50.000000      72.596100  50.262611   
C02     CNN          96.774194  86.363636      87.792751  75.437569   
        ensemble     83.870968  63.636364      74.910563  61.900950   
C03     CNN          94.594595  84.615385      89.962529  78.766324   
        ensemble     83.783784  96.153846      80.943071  81.987407   
U01     CNN          88.034188  79.487179      82.348574  81.441930   
        ensemble     80.341880  78.205128      76.664892  77.128975   
U03     CNN          85.294118  66.666667      87.242276  64.621099   
        ensemble     73.529412  44.444444      77.245481  50.062963   
U04     CNN          78.378378  56.000000      75.643038  54.338843   
        ensemble     72.972973  40.000000      63.959110  39.961982   
U05     CNN          80.000000  85.714286      86.207317  81.808801   
        ensemble     77.500000  78.571429      80.069576  79.784715   
U07     CNN         100.000000  66.666667      98.245440  73.387531   
        ensemble     85.714286  83.333333      78.123702  65.705911   
U12     CNN          77.777778  66.666667      84.620551  63.443760   
        ensemble     83.333333  75.000000      68.373241  61.267223   
U15     CNN          76.923077  70.000000      85.114059  67.660409   
        ensemble     84.615385  60.000000      75.649040  59.294925   
U16     CNN         100.000000  62.500000      92.993249  63.698050   
        ensemble     75.000000  75.000000      78.852871  76.663270   
U17     CNN          83.333333  77.777778      83.827358  70.280838   
        ensemble     83.333333  77.777778      73.315735  73.358998   

metric              roc_auc             
split                 train       test  
patient model                           
C01     CNN   

In [23]:
# Build per-model and overall summary rows with index = (Summary, Model)
summary_frames = []

for func in ['min', 'max', 'mean']:
    # Per-model summary
    for model in ['CNN', 'ensemble']:
        model_slice = n.xs(model, level='model')
        row = getattr(model_slice, func)(axis=0, numeric_only=True)
        row.name = (func, model)
        summary_frames.append(row.to_frame().T)

    # Overall summary
    row = getattr(n, func)(axis=0, numeric_only=True)
    row.name = (func, 'overall')
    summary_frames.append(row.to_frame().T)

summary_df = pd.concat(summary_frames)
summary_df

metric        best_threshold            eb_specificity             \
split                  train       test          train       test   
min  CNN           58.211178  34.802124      73.092182  52.773398   
     ensemble      52.697384  42.071170      56.927266  39.924036   
     overall       52.697384  34.802124      56.927266  39.924036   
max  CNN           96.700019  86.892021      96.551388  83.495247   
     ensemble      65.469843  62.252885      83.123035  81.036060   
     overall       96.700019  86.892021      96.551388  83.495247   
mean CNN           72.256507  61.105136      86.667793  67.813457   
     ensemble      57.572070  54.723848      70.856188  62.419768   
     overall       64.914288  57.914492      78.761991  65.116612   

metric        rel_szrs_pred            event_based_f1               roc_auc  \
split                 train       test          train       test      train   
min  CNN          76.923077  54.166667      75.643038  54.338843  78.293326   
     ensemble     72.972973  40.000000      63.959110  39.961982  65.555239   
     overall      72.972973  40.000000      63.959110  39.961982  65.555239   
max  CNN         100.000000  86.363636      98.245440  81.808801  96.910532   
     ensemble     85.714286  96.153846      80.943071  81.987407  85.929523   
     overall     100.000000  96.153846      98.245440  81.987407  96.910532   
mean CNN          86.710119  71.385411      86.338632  69.290417  89.530601   
     ensemble     80.774123  68.510194      75.058615  64.781661  76.332762   
     overall      83.742121  69.947802      80.698624  67.036039  82.931681   

metric                    
split               test  
min  CNN       53.353268  
     ensemble  37.986387  
     overall   37.986387  
max  CNN       85.317380  
     ensemble  85.490809  
     overall   85.490809  
mean CNN       68.373946  
     ensemble  65.602764  
     overall   66.988355

In [24]:
# Mark rows where p_hanley_mcneil is insignificant with asterisks
def convert_to_str(df):
    return df.round().astype(int).astype(str)


s = convert_to_str(n)
s['roc_auc'] = s['roc_auc'].where(p_hm < ALPHA, s['roc_auc'] + '*')
s

metric           best_threshold      eb_specificity      rel_szrs_pred       \
split                     train test          train test         train test   
patient model                                                                 
C01     CNN                  73   35             85   59            79   54   
        ensemble             55   42             63   51            85   50   
C02     CNN                  59   50             80   67            97   86   
        ensemble             60   56             68   60            84   64   
C03     CNN                  78   70             86   74            95   85   
        ensemble             65   62             78   71            84   96   
U01     CNN                  58   63             77   83            88   79   
        ensemble             57   54             73   76            80   78   
U03     CNN                  78   67             89   63            85   67   
        ensemble             57   53             81   57            74   44   
U04     CNN                  60   56             73   53            78   56   
        ensemble             53   51             57   40            73   40   
U05     CNN                  76   60             93   78            80   86   
        ensemble             59   59             83   81            78   79   
U07     CNN                  74   70             97   82           100   67   
        ensemble             55   54             72   54            86   83   
U12     CNN                  67   47             93   61            78   67   
        ensemble             56   56             58   52            83   75   
U15     CNN                  97   87             95   65            77   70   
        ensemble             56   54             68   59            85   60   
U16     CNN                  81   75             87   65           100   62   
        ensemble             56   56             83   78            75   75   
U17     CNN                  67   53             84   64            83   78   
        ensemble             61   60             65   69            83   78   

metric           event_based_f1      roc_auc       
split                     train test   train test  
patient model                                      
C01     CNN                  82   57      88   61  
        ensemble             73   50      80   59  
C02     CNN                  88   75      89   80  
        ensemble             75   62      77   66  
C03     CNN                  90   79      92   85  
        ensemble             81   82      86   85  
U01     CNN                  82   81      86   82  
        ensemble             77   77      80   79  
U03     CNN                  87   65      90   57  
        ensemble             77   50      76  50*  
U04     CNN                  76   54      78  53*  
        ensemble             64   40      66  38*  
U05     CNN                  86   82      91   81  
        ensemble             80   80      80   76  
U07     CNN                  98   73      92   69  
        ensemble             78   66      73   69  
U12     CNN                  85   63      93   61  
        ensemble             68   61      73   66  
U15     CNN                  85   68      88   60  
        ensemble             76   59      76   58  
U16     CNN                  93   64      97   61  
        ensemble             79   77      75   61  
U17     CNN                  84   70      90   69  
        ensemble             73   73      75   79

In [38]:
# Add summary frames
index_names = s.index.names
r = pd.concat([s, convert_to_str(summary_df)])  # r: resulting dataframe
r.index.names = index_names

# Rename
r = r.rename_axis(index={'patient': r'\textbf{Patient}', 'model': r'\textbf{Model}'},
                  columns={'metric': r'\textbf{Metric:}', 'split': r'\textbf{Set:}'})
r: DataFrame = r.rename(
    columns={
        # 'best_threshold': r'\textbf{Thresh.}',
        'best_threshold': r'\makecell{\textbf{Optimal}\\\textbf{Threshold}}',
        # 'rel_tifw': r'\textbf{Rel. TIFW}',
        # 'eb_specificity': r'\textbf{EB Spec.}',
        'eb_specificity': r'\makecell{\textbf{EB}\\\textbf{Specificity}}',
        # 'rel_szrs_pred': r'\textbf{EB Sens.}',
        'rel_szrs_pred': r'\makecell{\textbf{EB}\\\textbf{Sensitivity}}',
        # 'event_based_f1': r'\textbf{EB Score}',
        'event_based_f1': r'\makecell{\textbf{EB}\\\textbf{Score}}',
        'precision': r'\textbf{Precision}',
        'recall': r'\textbf{Recall}',
        # 'roc_auc': r'\textbf{ROC AUC}',
        'roc_auc': r'\makecell{\textbf{ROC}\\\textbf{AUC}}',
        'p_hanley_mcneil': r'\boldsymbol{$p_{\mathrm{HM}}$}',
        # This makes the table too wide
        'train': r'\textbf{Train}',
        'test': r'\textbf{Test}',
    },
    index={
        'ensemble': 'Ensemble',
        'overall': 'Overall',
        'min': 'Min',
        'max': 'Max',
        'mean': 'Mean'
    }
)
r

\textbf{Metric:}                \makecell{\textbf{Optimal}\\\textbf{Threshold}}  \
\textbf{Set:}                                                    \textbf{Train}   
\textbf{Patient} \textbf{Model}                                                   
C01              CNN                                                         73   
                 Ensemble                                                    55   
C02              CNN                                                         59   
                 Ensemble                                                    60   
C03              CNN                                                         78   
                 Ensemble                                                    65   
U01              CNN                                                         58   
                 Ensemble                                                    57   
U03              CNN                                                         78   
                 Ensemble                                                    57   
U04              CNN                                                         60   
                 Ensemble                                                    53   
U05              CNN                                                         76   
                 Ensemble                                                    59   
U07              CNN                                                         74   
                 Ensemble                                                    55   
U12              CNN                                                         67   
                 Ensemble                                                    56   
U15              CNN                                                         97   
                 Ensemble                                                    56   
U16              CNN                                                         81   
                 Ensemble                                                    56   
U17              CNN                                                         67   
                 Ensemble                                                    61   
Min              CNN                                                         58   
                 Ensemble                                                    53   
                 Overall                                                     53   
Max              CNN                                                         97   
                 Ensemble                                                    65   
                 Overall                                                     97   
Mean             CNN                                                         72   
                 Ensemble                                                    58   
                 Overall                                                     65   

\textbf{Metric:}                               \
\textbf{Set:}                   \textbf{Test}   
\textbf{Patient} \textbf{Model}                 
C01              CNN                       35   
                 Ensemble                  42   
C02              CNN                       50   
                 Ensemble                  56   
C03              CNN                       70   
                 Ensemble                  62   
U01              CNN                       63   
                 Ensemble                  54   
U03              CNN                       67   
                 Ensemble                  53   
U04              CNN                       56   
                 Ensemble                  51   
U05              CNN                       60   
                 Ensemble                  59   
U07              CNN                       70   
                 Ensemble                  54   
U12              CNN                       47   
                 Ensemble      

In [40]:
l = r.to_latex(
    multirow=True,
    multicolumn=True,
    multicolumn_format='c',
    column_format='ll rr rr rr rr rr rr'
)

# ---- Shade every second multiindex row
header, table = l.split('\n\\midrule\n')
table, footer = table.split('\n\\cline{1-12}\n\\bottomrule\n')

groups = table.split('\n\\cline{1-12}\n')  # split into multiindex groups

for i, group in enumerate(groups):
    rowcolor = '\\rowcolor{gray!10} ' if i % 2 == 0 else '\\rowcolor{white}   '

    lines = group.split('\n')
    for j, line in enumerate(lines):
        lines[j] = rowcolor + line
    groups[i] = '\n'.join(lines)

# for g in groups:
#     print('-------')
#     print(g)
#     print('-------')
#
l = header + '\n\\midrule\n' + '\n'.join(groups) + '\n\\bottomrule\n' + footer

# ---- Put a separator before Min row
l = l.replace(
    '\\rowcolor{gray!10} \\multirow[t]{3}{*}{Min}',
    '\\midrule\n\\rowcolor{gray!10} \\multirow[t]{3}{*}{Min}',
)

# ---- Add cmidrules to indicate multicolumn spans
l = l.replace(
    # '{\\textbf{ROC AUC}} \\\\',
    # '{\\textbf{ROC AUC}} \\\\\n\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}\\cmidrule(lr){7-8}\\cmidrule(lr){9-10}\\cmidrule(lr){11-12}'
    '\\textbf{AUC}}} \\\\',
    '\\textbf{AUC}}} \\\\\n\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}\\cmidrule(lr){7-8}\\cmidrule(lr){9-10}\\cmidrule(lr){11-12}'
)

# out_path.write_text(l, encoding="utf-8")
print(l)

\begin{tabular}{ll rr rr rr rr rr rr}
\toprule
 & \textbf{Metric:} & \multicolumn{2}{c}{\makecell{\textbf{Optimal}\\\textbf{Threshold}}} & \multicolumn{2}{c}{\makecell{\textbf{EB}\\\textbf{Specificity}}} & \multicolumn{2}{c}{\makecell{\textbf{EB}\\\textbf{Sensitivity}}} & \multicolumn{2}{c}{\makecell{\textbf{EB}\\\textbf{Score}}} & \multicolumn{2}{c}{\makecell{\textbf{ROC}\\\textbf{AUC}}} \\
\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}\cmidrule(lr){9-10}\cmidrule(lr){11-12}
 & \textbf{Set:} & \textbf{Train} & \textbf{Test} & \textbf{Train} & \textbf{Test} & \textbf{Train} & \textbf{Test} & \textbf{Train} & \textbf{Test} & \textbf{Train} & \textbf{Test} \\
\textbf{Patient} & \textbf{Model} &  &  &  &  &  &  &  &  &  &  \\
\midrule
\rowcolor{gray!10} \multirow[t]{2}{*}{C01} & CNN & 73 & 35 & 85 & 59 & 79 & 54 & 82 & 57 & 88 & 61 \\
\rowcolor{gray!10}  & Ensemble & 55 & 42 & 63 & 51 & 85 & 50 & 73 & 50 & 80 & 59 \\
\rowcolor{white}   \multirow[t]{2}{*}{C02} & CNN & 59 & 50 & 80 